# PKG counterparty metrics — `pkg_cpty_metrics.ipynb`

One row per **counterparty entity** (`ent_l2` from `pkg_cpty_entity_map_v2`) per month, seen through PNC customers.

**Eligibility (simple, explainable):** an entity gets the **full metric set** in a month when it moved at least
`ELIG_AMT` (10,000 dollars) with PNC customers, both directions, over the trailing `ELIG_WINDOW` (12) months including that
month. Every entity active in the month (or last month) gets the **slim** columns: identity, eligibility and flow.
Name class, cpty type and hub flags are **labels** on the row, never used to decide eligibility.

**Direction is the counterparty's view** (the opposite of `pkg_cust_metrics_monthly`): `_in` = the counterparty
receives from PNC customers (edge `dst_kind = CPTY`); `_out` = the counterparty pays PNC customers (`src_kind = CPTY`).

**Outputs** in `bdahd01p_dlcdi1_cdi_tm` (external parquet):

| Table | Grain | Built in |
|---|---|---|
| `pkg_cpty_entity_dim` | one row per entity: name, class, type, ids, banks, lifetime flow, PNC-customer name match | §2 |
| `pkg_cpty_metrics_monthly` | one row per entity × month, partitioned by `month` | §3 |
| `pkg_cpty_metrics_dictionary` | one row per column: family, set (slim/full), direction, definition | §4 |

**Resume.** Run all cells; it always continues where it stopped.
- A month is *done* when its partition has `_SUCCESS` **and** the marker of the current configuration
  (`_pkg_sig_<SIG>`). Months that are not done are built in calendar order (churn and eligibility need the month
  before); months already done are skipped. Rerunning only the §3b cell also resumes.
- Inside a month each step writes its result to disk; an interrupted month restarts from the step that failed.
- A month is written to a staging folder, reconciled exactly against `pkg_edge_monthly`, then renamed into place,
  so a half-written month never looks done.
- The signature `SIG` covers the settings that change the numbers (eligibility, churn windows, minimums, metric
  version) **and the content of `pkg_cpty_entity_map_v2`, `pkg_fi_routing`, `pkg_fi_name_map`** (file fingerprint).
  A new signature makes all months stale: they are rebuilt from `HISTORY_START` with fresh state.
- When the **edge tables** were rebuilt for some months, set `REBUILD_FROM = "YYYY-MM"`: that month and every later
  month are deleted first, then rebuilt (needs the churn state of the month before, i.e. within `KEEP_STATE`). `MAX_MONTHS_PER_RUN` stops after N months so the backfill can run in chunks.
- Churn state is kept for the last `KEEP_STATE` months only.

**Run order:** `pkg_cpty_entity_map_v2.ipynb` → `pkg_fi_canonical.ipynb` → this notebook (Run All). Spark settings
are the heavy-job ones (8g executors, 2 cores, 3g overhead, skew join on). First time: set `MAX_MONTHS_PER_RUN = 1`,
check the minutes in §3 and the sizes in §6, then run the rest in chunks.

**Disk:** churn state is the largest item (one row per entity × customer × direction seen in the last 12 months,
`KEEP_STATE` months kept); per-month scratch is deleted after the month is published.

## §0 Setup

In [ ]:
# §0a  Session and compat shims
import os, json, time, math, hashlib, warnings, datetime as dt, html as _html
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
import numpy as np
import pandas as pd
with warnings.catch_warnings():                 # hasattr on removed aliases warns in NumPy 2
    warnings.simplefilter("ignore")
    for _alias, _target in [("object0", np.object_), ("bool8", np.bool_), ("int0", np.intp),
                            ("uint0", np.uintp), ("bool", np.bool_), ("object", object)]:   # np.object: non-Arrow toPandas
        if not hasattr(np, _alias):
            setattr(np, _alias, _target)
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
from IPython.display import display, HTML
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.utils import AnalysisException

SHUFFLE_PARTITIONS = 2000
# Executor memory per core matters here (hub entities, million-row joins): 8g × 2 cores = 4g per task.
spark = (SparkSession.builder
         .appName("pkg_cpty_metrics")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "8g")
         .config("spark.executor.cores", "2")
         .config("spark.executor.memoryOverhead", "3g")
         .config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g")
         .config("spark.dynamicAllocation.enabled", "true")
         .config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.sql.adaptive.skewJoin.enabled", "true")
         .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "32m")
         .enableHiveSupport()
         .getOrCreate())
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
spark.conf.set("spark.sql.ansi.enabled", "false")      # ratios with a zero denominator are NULL, not an error
print(f"Spark {spark.version} | pandas {pd.__version__} | numpy {np.__version__}")

In [ ]:
# §0b  Config
DB           = "bdahd01p_dlcdi1_cdi_tm"
T_EDGE_M     = f"{DB}.pkg_edge_monthly"
T_EDGE_D     = f"{DB}.pkg_edge_daily"
T_CPTY_MONTH = f"{DB}.pkg_cpty_month"
T_CPTY_DIM   = f"{DB}.pkg_cpty_dim"
T_MAP        = f"{DB}.pkg_cpty_entity_map_v2"
T_MAP_V1     = f"{DB}.pkg_cpty_entity_map"           # only read in §1b to locate duplicate ids
T_CUST_DIM   = f"{DB}.pkg_cust_dim"
T_FI_ROUTING = f"{DB}.pkg_fi_routing"
T_FI_NAMEMAP = f"{DB}.pkg_fi_name_map"
T_ENT_DIM    = f"{DB}.pkg_cpty_entity_dim"          # written in §2
T_OUT        = f"{DB}.pkg_cpty_metrics_monthly"     # written in §3
T_DICT       = f"{DB}.pkg_cpty_metrics_dictionary"  # written in §4

HISTORY_START = "2024-01"      # first month of the edge tables (churn and eligibility history start here)
LAST_MONTH    = "2026-08"      # last complete month to build

ELIG_AMT    = 10_000           # full metrics when trailing-window $ (both directions) ≥ this
ELIG_WINDOW = 12               # months in the trailing window, including the month itself
LOOKBACK    = 12               # churn memory in months (new = not seen in the previous LOOKBACK months)
REC_MIN, REC_WIN = 4, 6        # recurring = present in ≥ REC_MIN of the previous REC_WIN months
DEP_MIN     = 0.5              # n_cust_dep50: counterparty is ≥ this share of the customer's external flow
NAICS_MIN   = 5                # sector metrics need ≥ this many PNC customers with a valid NAICS
HUB_Q       = [0.99, 0.999, 0.9999]   # hub_tier cut-offs (degree or strength) among eligible entities
HUB_MED_MULT = 15              # hub_med15: degree ≥ this multiple of the median eligible degree
USE_DAILY   = True             # active_days_* from pkg_edge_daily (adds a few minutes per month)
REQUIRE_CANONICAL_FI = True    # fail if pkg_fi_routing / pkg_fi_name_map are missing (else raw FI names)

REBUILD_FROM       = None      # e.g. "2025-03": delete and rebuild that month and every later month
MAX_MONTHS_PER_RUN = None      # e.g. 6: stop after this many months (run the backfill in chunks)
FORCE_ENT_DIM      = False     # rebuild pkg_cpty_entity_dim even if it matches the current inputs
KEEP_STATE         = 3         # months of churn state kept on disk (≥ 2)
OUT_FILES          = 16        # parquet files per output month
CLEAN_OTHER_SIGS   = False     # §6: delete state / scratch folders of other configurations
METRICS_VERSION    = "cpty_v1.2"  # bump when columns or definitions change (forces a rebuild)
ENT_DIM_VERSION    = "dim_v2"     # bump to rebuild pkg_cpty_entity_dim only
RUNLOG             = "pkg_cpty_metrics_runlog.csv"

# Legal-form words removed before matching counterparty names to PNC customer names (pkg_cpty_entity_dim)
LEGAL = ["INC", "INCORPORATED", "LLC", "LLP", "LP", "LTD", "LIMITED", "CORP", "CORPORATION", "CO", "COMPANY", "PC",
         "PLLC", "PA", "PLC", "NA", "THE", "DBA"]

assert KEEP_STATE >= 2 and LOOKBACK <= 31 and 1 <= ELIG_WINDOW <= 24
print(f"{HISTORY_START} → {LAST_MONTH} | eligible: ≥ ${ELIG_AMT:,} over {ELIG_WINDOW} months | signature in §0c")

In [ ]:
# §0c  Helpers: months, file system, display, canonical FI, dictionary
def month_add(m, k):
    t = int(m[:4]) * 12 + int(m[5:7]) - 1 + k
    return f"{t // 12:04d}-{t % 12 + 1:02d}"

def month_range(a, b):
    out, m = [], a
    while m <= b:
        out.append(m)
        m = month_add(m, 1)
    return out

def prev_month(m):
    return month_add(m, -1)

# ---- file system (Hadoop FileSystem through the JVM; works for hdfs:// and local paths) ----
_jvm, _hconf = spark._jvm, spark._jsc.hadoopConfiguration()
def _fs(p):
    path = _jvm.org.apache.hadoop.fs.Path(p)
    return path.getFileSystem(_hconf), path
def exists(p):
    fs, path = _fs(p)
    return fs.exists(path)
def delete(p):
    fs, path = _fs(p)
    return bool(fs.exists(path) and fs.delete(path, True))
def mkdirs(p):
    fs, path = _fs(p)
    fs.mkdirs(path)
def rename(src, dst):
    fs, s = _fs(src)
    _, d = _fs(dst)
    if not fs.rename(s, d):
        raise RuntimeError(f"rename failed: {src} → {dst}")
def touch(p):
    fs, path = _fs(p)
    fs.create(path, True).close()
def ls_dirs(p):
    fs, path = _fs(p)
    if not fs.exists(path):
        return []
    return [st.getPath().getName() for st in fs.listStatus(path) if st.isDirectory()]
def path_gb(p):
    fs, path = _fs(p)
    return fs.getContentSummary(path).getLength() / 1e9 if fs.exists(path) else 0.0
def db_location(db):
    for r in spark.sql(f"DESCRIBE DATABASE EXTENDED {db}").collect():
        if str(r[0]).strip().lower() == "location":
            return str(r[1]).rstrip("/")
    raise RuntimeError(f"no Location row in DESCRIBE DATABASE {db}")
def table_exists(t):
    try:
        spark.table(t).columns
        return True
    except AnalysisException:
        return False

def table_fp(t):
    # content fingerprint of a table's files: count, bytes, latest modification (changes when the table is rewritten)
    loc = next(str(r[1]) for r in spark.sql(f"DESCRIBE FORMATTED {t}").collect()
               if str(r[0]).strip().lower() == "location")
    fs, path = _fs(loc)
    it, n, size, mt = fs.listFiles(path, True), 0, 0, 0
    while it.hasNext():
        st = it.next()
        if not st.getPath().getName().startswith((".", "_")):
            n, size, mt = n + 1, size + st.getLen(), max(mt, st.getModificationTime())
    return f"{n}:{size}:{mt}"

HAVE_CANON = table_exists(T_FI_ROUTING) and table_exists(T_FI_NAMEMAP)
# Signature of the configuration: settings that change the numbers + content of the entity map and canonical FI
# tables. A new signature makes every month stale (rebuilt from HISTORY_START with fresh churn state).
FP = {"map": table_fp(T_MAP),
      "fi": [table_fp(T_FI_ROUTING), table_fp(T_FI_NAMEMAP)] if HAVE_CANON else None}
SIG = hashlib.md5(json.dumps({"v": METRICS_VERSION, "map": T_MAP, "fp": FP, "start": HISTORY_START, "elig": ELIG_AMT,
                              "win": ELIG_WINDOW, "lb": LOOKBACK, "rec": [REC_MIN, REC_WIN], "dep": DEP_MIN,
                              "naics": NAICS_MIN, "hub": [HUB_Q, HUB_MED_MULT], "daily": USE_DAILY},
                             sort_keys=True).encode()).hexdigest()[:10]

LOC       = db_location(DB)
P_OUT     = f"{LOC}/pkg_cpty_metrics_monthly"
P_ENT_DIM = f"{LOC}/pkg_cpty_entity_dim"
P_DICT    = f"{LOC}/pkg_cpty_metrics_dictionary"
P_STATE_ROOT, P_WORK_ROOT = f"{LOC}/_pkg_cpty_metrics_state", f"{LOC}/_tmp_pkg_cpty_metrics"
P_STATE, P_WORK = f"{P_STATE_ROOT}/{SIG}", f"{P_WORK_ROOT}/{SIG}"
SIG_MARK = f"_pkg_sig_{SIG}"

def out_path(m):
    return f"{P_OUT}/month={m}"
def state_path(kind, m):
    return f"{P_STATE}/{kind}/month={m}"
def work_path(m, name):
    return f"{P_WORK}/{m}/{name}"
def month_status(m):
    p = out_path(m)
    if not exists(p + "/_SUCCESS"):
        return "missing"
    return "done" if exists(f"{p}/{SIG_MARK}") else "stale"

# ---- display: one output per cell ----
def _fmt_val(v, kind):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    if v == 0 or abs(v) < 5e-7:
        return "0"
    if kind == "pct":
        return "0" if v == 0 else (f"{v:.3f}" if abs(v) < 1 else f"{v:.2f}")
    if kind == "f1":
        return f"{v:,.1f}"
    if kind == "int":
        return f"{v:,.0f}"
    return f"{v:,.3f}" if abs(v) < 10 else f"{v:,.1f}"

def fmt_df(df):
    out = df.copy()
    out.columns.name = None
    for c in out.columns:
        s = out[c]
        if pd.api.types.is_bool_dtype(s):
            continue
        if not pd.api.types.is_numeric_dtype(s):
            out[c] = s.where(s.notna(), "")
            continue
        name = str(c)
        kind = ("pct" if "%" in name else "f1" if name.startswith("$M") or name.startswith("$B") or name == "min"
                else "int" if pd.api.types.is_integer_dtype(s) or bool((s.dropna() % 1 == 0).all()) else "f2")
        out[c] = [_fmt_val(v, kind) for v in s]
    return out

_CSS = ("<style>.pkgt{border-collapse:collapse;font-size:12px;margin-bottom:4px}"
        ".pkgt th,.pkgt td{padding:2px 9px;border-bottom:1px solid #e3e2de;text-align:right;white-space:nowrap}"
        ".pkgt th{background:#e3e2de;color:#0b0b0b !important;font-weight:600}.pkgn{color:#8a8984;margin:2px 0}</style>")

def note(text):
    return ("__note__", text)

def render(*blocks):
    parts = []
    for b in blocks:
        if b is None:
            continue
        if isinstance(b, str):
            parts.append(f"<div style='margin:10px 0 2px;font-size:14px'><b>{_html.escape(b)}</b></div>")
        elif b[0] == "__note__":
            parts.append(f"<div class='pkgn'>{_html.escape(b[1])}</div>")
        else:
            parts.append(f"<div style='margin:12px 0 3px'><b>{_html.escape(b[0])}</b></div>")
            parts.append(fmt_df(b[1]).to_html(index=False, escape=True, border=0, classes="pkgt"))
    return HTML(_CSS + "<div style='font-family:\"IBM Plex Sans\",Arial,sans-serif'>" + "".join(parts) + "</div>")

def show(*blocks):
    display(render(*blocks))

# ---- canonical FI (pkg_fi_canonical.ipynb) ----
if REQUIRE_CANONICAL_FI and not HAVE_CANON:
    raise RuntimeError(f"{T_FI_ROUTING} / {T_FI_NAMEMAP} missing: run pkg_fi_canonical.ipynb first "
                       "(or set REQUIRE_CANONICAL_FI = False to use raw FI names)")

def canon_fi(df, id_col, fi_col, out="fi_c"):
    # canonical bank: routing prefix of the id → pkg_fi_routing, else raw FI name → pkg_fi_name_map, else raw name
    if not HAVE_CANON:
        return df.withColumn(out, F.col(fi_col))
    rt = F.broadcast(spark.table(T_FI_ROUTING).where(F.col("fi_canonical").isNotNull())
                          .select(F.col("routing").alias("_rt"), F.col("fi_canonical").alias("_fc_rt")))
    nm = F.broadcast(spark.table(T_FI_NAMEMAP).select(F.col("fi_raw").alias("_fr"), F.col("fi_canonical").alias("_fc_nm")))
    return (df.withColumn("_rt", F.regexp_extract(F.col(id_col), r"^(\d{9})-", 1))
              .join(rt, "_rt", "left")
              .join(nm, F.col(fi_col) == F.col("_fr"), "left")
              .withColumn(out, F.coalesce("_fc_rt", "_fc_nm", fi_col))
              .drop("_rt", "_fc_rt", "_fr", "_fc_nm"))

def simple_key(c):
    k = F.regexp_replace(F.upper(c), r"[^A-Z0-9]+", " ")
    k = F.regexp_replace(F.trim(k), r"\b([A-Z]) (?=[A-Z]\b)", "$1")
    k = F.regexp_replace(k, r"\b(" + "|".join(sorted(LEGAL, key=len, reverse=True)) + r")\b", " ")
    return F.trim(F.regexp_replace(k, r"\s+", " "))

# ---- metric dictionary: column order, types and definitions of pkg_cpty_metrics_monthly ----
DEC = "decimal(28,2)"
RAILS = ["ach", "wire", "rtp_prt", "rtp_p2p", "check", "pcard"]
RAIL_NUMS = {"ach": ["1", "2", "3"], "wire": ["4"], "rtp_prt": ["5"], "rtp_p2p": ["6"], "check": ["7"], "pcard": ["9"]}
DIRS = ["in", "out"]
DIR_TXT = {"in": "PNC customers → counterparty", "out": "counterparty → PNC customers"}
DICT = []
def _d(col, fam, st, typ, definition, d=""):
    DICT.append({"column": col, "family": fam, "set": st, "direction": d, "type": typ, "definition": definition})
def _dd(base, fam, typ, definition):
    for d in DIRS:
        _d(f"{base}_{d}", fam, "full", typ, definition, d)

_d("ent_id", "identity", "slim", "string", "Counterparty entity id (ent_l2 of pkg_cpty_entity_map_v2); key to pkg_cpty_entity_dim")
_d("ent_name", "identity", "slim", "string", "Dominant counterparty name of the entity (by lifetime $)")
_d("name_class", "identity", "slim", "string", "Typer v2 class of the dominant id (business / person / unknown / numeric_name / bank / placeholder / no_name; not_in_map = id missing from the entity map); a label, not used for eligibility")
_d("cpty_type", "identity", "slim", "string", "Dominant cpty_type (CPTY / P2P_CPTY / MERCHANT / NON_PNC_ACH_ORIGINATOR)")
_d("id_src", "identity", "slim", "string", "Id source of the dominant id: A = account id, B = name-hash id (3B/3C/7B/9B)")
_d("n_ids_active", "identity", "slim", "bigint", "Raw counterparty ids of the entity active this month")
_d("fi_dom", "identity", "slim", "string", "Dominant holding bank this month (canonical FI name, by $)")
_d("amt_12m", "eligibility", "slim", DEC, "PNC-visible $ in both directions over the trailing ELIG_WINDOW months, this month included")
_d("active_months_12m", "eligibility", "slim", "bigint", "Months with activity in the trailing window")
_d("history_months", "eligibility", "slim", "int", "Months of history available in the window (fewer than ELIG_WINDOW in the first year)")
_d("eligible", "eligibility", "slim", "boolean", "amt_12m ≥ ELIG_AMT: full metric set filled; otherwise full columns are NULL")
for d in DIRS:
    _d(f"amt_{d}", "flow", "slim", DEC, f"$ this month, {DIR_TXT[d]}", d)
_d("amt_all", "flow", "slim", DEC, "amt_in + amt_out")
for d in DIRS:
    _d(f"n_txn_{d}", "flow", "slim", "bigint", f"Transactions this month, {DIR_TXT[d]}", d)
for d in DIRS:
    _d(f"n_cust_{d}", "breadth", "slim", "bigint", f"Distinct PNC customers (mdm_id), {DIR_TXT[d]}", d)
_d("n_cust_all", "breadth", "slim", "bigint", "Distinct PNC customers in either direction (degree)")
_dd("avg_ticket", "flow", "double", "amt / n_txn")
_dd("active_days", "flow", "bigint", "Distinct days with a transaction across the entity's ids (pkg_edge_daily; NULL when USE_DAILY = False)")
_dd("amt_self", "flow", DEC, "$ on self edges (the PNC customer's own account at another bank)")
_d("net_in", "flow", "full", DEC, "amt_in − amt_out: net $ the counterparty receives from PNC customers")
_d("two_way_ratio", "flow", "full", "double", "min(amt_in, amt_out) / max(amt_in, amt_out); 0 = one-way")
_dd("n_cust_tm", "breadth", "bigint", "Distinct TM customers")
_dd("n_family", "breadth", "bigint", "Distinct customer families (master_pwr_id; customers without one count individually)")
_dd("tm_share", "breadth", "double", "$ share with TM customers")
_dd("org_share", "breadth", "double", "$ share with organisation customers (party_type O)")
_dd("top1_share", "concentration", "double", "$ share of the largest PNC customer")
_dd("hhi", "concentration", "double", "Herfindahl index of $ across PNC customers (1 = one customer)")
_dd("top_cust", "concentration", "string", "mdm_id of the largest PNC customer by $")
_dd("top_cust_name", "concentration", "string", "customer_name of top_cust (pkg_cust_dim)")
_dd("n_naics2", "sector", "bigint", "Distinct NAICS2 sectors of its PNC customers (valid NAICS only; NULL below NAICS_MIN customers)")
_dd("naics2_top", "sector", "string", "Dominant NAICS2 sector of its PNC customers by $ (same NULL rule)")
_dd("naics2_top_desc", "sector", "string", "NAICS 2-digit sector title of naics2_top")
_dd("naics2_top_share", "sector", "double", "$ share of the dominant NAICS2 sector (same NULL rule)")
_dd("naics2_entropy", "sector", "double", "Shannon entropy of $ across NAICS2 sectors, natural log (same NULL rule)")
_dd("dep_wavg", "dependency", "double", "$-weighted mean, over its PNC customers, of its share of each customer's external flow in this direction")
_dd("n_cust_dep50", "dependency", "bigint", "PNC customers for whom it is ≥ DEP_MIN of their external flow in this direction")
_dd("new_cust", "churn", "bigint", "PNC customers present now, not seen in the previous LOOKBACK months")
_dd("reactivated_cust", "churn", "bigint", "PNC customers present now, seen in the previous LOOKBACK months but not last month")
_dd("retained_cust", "churn", "bigint", "PNC customers present last month and now")
_dd("lost_cust", "churn", "bigint", "PNC customers present last month, not now")
_dd("retention", "churn", "double", "retained / (retained + lost)")
_dd("new_amt_share", "churn", "double", "$ share from new customers this month")
_dd("lost_amt_share", "churn", "double", "Last month's $ of lost customers / last month's total $")
_dd("rec_standing", "churn", "bigint", "Recurring customers: present in ≥ REC_MIN of the previous REC_WIN months")
_dd("rec_broken", "churn", "bigint", "Recurring customers absent this month")
_dd("rec_broken_amt_share", "churn", "double", "Last monthly $ of broken recurring customers / last monthly $ of all recurring")
for r in RAILS:
    _dd(f"share_{r}", "rails", "double", f"$ share on {r.upper()} (category {'/'.join(RAIL_NUMS[r])})")
_dd("n_rails", "rails", "bigint", "Rails with activity")
_d("n_fi", "banks", "full", "bigint", "Holding banks of the entity's active ids this month (canonical FI)")
_d("top_fi_share", "banks", "full", "double", "$ share of the dominant holding bank")
_d("fi_new", "banks", "full", "bigint", "Holding banks not seen in the previous LOOKBACK months")
_d("fi_lost", "banks", "full", "bigint", "Holding banks present last month, not now")
_d("hub_tier", "hub", "full", "int", "0–3: degree (n_cust_all) or strength (amt_all) ≥ P99 / P99.9 / P99.99 among eligible entities this month")
_d("deg_x_median", "hub", "full", "double", "n_cust_all / median n_cust_all of eligible entities this month")
_d("hub_med15", "hub", "full", "boolean", "deg_x_median ≥ HUB_MED_MULT")
DICT_DF = pd.DataFrame(DICT)
DICT_DF["empty_value"] = np.where(DICT_DF.set == "slim", "0 / text always filled",
                                  np.where(DICT_DF.type.isin(["bigint", "int", DEC]), "0 when nothing happened",
                                           "NULL when the denominator is 0"))
DICT_DF.loc[DICT_DF.column.str.startswith(("n_naics2", "naics2_")), "empty_value"] = "NULL below NAICS_MIN valid-NAICS customers"
DICT_DF.loc[DICT_DF.column.str.startswith(("top_cust", "fi_dom", "ent_name", "name_class", "cpty_type", "id_src")),
            "empty_value"] = "NULL when unknown / no activity"
NAICS2_DESC = {"11": "Agriculture, Forestry, Fishing and Hunting", "21": "Mining, Quarrying, and Oil and Gas Extraction",
               "22": "Utilities", "23": "Construction", "31": "Manufacturing", "32": "Manufacturing",
               "33": "Manufacturing", "42": "Wholesale Trade", "44": "Retail Trade", "45": "Retail Trade",
               "48": "Transportation and Warehousing", "49": "Transportation and Warehousing", "51": "Information",
               "52": "Finance and Insurance", "53": "Real Estate and Rental and Leasing",
               "54": "Professional, Scientific, and Technical Services",
               "55": "Management of Companies and Enterprises",
               "56": "Administrative and Support and Waste Management and Remediation Services",
               "61": "Educational Services", "62": "Health Care and Social Assistance",
               "71": "Arts, Entertainment, and Recreation", "72": "Accommodation and Food Services",
               "81": "Other Services (except Public Administration)", "92": "Public Administration"}
OUT_COLS = list(DICT_DF.column)
FULL_COLS = list(DICT_DF.loc[DICT_DF.set == "full", "column"])
assert len(OUT_COLS) == len(set(OUT_COLS))
print(f"signature {SIG} | {len(OUT_COLS)} columns ({len(FULL_COLS)} full-set) | output {P_OUT}")

## §1 Inputs and build plan

Checks columns and id types, the edge partitions for every month to build, and the canonical FI tables. Then lists
every month with its status: `done` (current configuration), `stale` (built with another configuration), `missing`.

In [ ]:
# §1  Input checks and plan
def need(table, cols):
    dt_ = dict(spark.table(table).dtypes)
    miss = [c for c in cols if c not in dt_]
    assert not miss, f"{table}: missing {miss}; has {sorted(dt_)}"
    return dt_

def pick(dt_, table, candidates, required=True):
    for c in candidates:
        if c in dt_:
            return c
    assert not required, f"{table}: none of {candidates}; has {sorted(dt_)}"
    return None

def partitions(table):
    out = set()
    for r in spark.sql(f"SHOW PARTITIONS {table}").collect():
        for kv in r[0].split("/"):
            if kv.startswith("month="):
                out.add(kv.split("=", 1)[1])
    return out

edt = need(T_EDGE_M, ["src_id", "src_kind", "dst_id", "dst_kind", "category", "is_self", "amount", "n_txn", "month"])
mdt = need(T_MAP, ["cpty_id", "ent_l2", "name_class", "type_reason", "cpty_id_src"])
cmt = need(T_CPTY_MONTH, ["cpty_id", "fi", "amount", "month"])
ddt = need(T_CPTY_DIM, ["cpty_id"])
udt = need(T_CUST_DIM, ["mdm_id", "is_tm"])
if USE_DAILY:
    need(T_EDGE_D, ["txn_date", "src_id", "src_kind", "dst_id", "dst_kind", "month"])
bad = {f"{t}.{c}": d[c] for t, d, cols in [(T_EDGE_M, edt, ["src_id", "dst_id"]), (T_MAP, mdt, ["cpty_id", "ent_l2"]),
                                           (T_CPTY_DIM, ddt, ["cpty_id"]), (T_CUST_DIM, udt, ["mdm_id"])]
       for c in cols if d[c] != "string"}
assert not bad, f"id columns must be string: {bad}"

DIM = {"name": pick(ddt, T_CPTY_DIM, ["cpty_name", "dominant_name", "name"]),
       "fi": pick(ddt, T_CPTY_DIM, ["fi", "cpty_fin_entity_name", "dominant_fi"]),
       "cpty_type": pick(ddt, T_CPTY_DIM, ["cpty_type", "dominant_cpty_type"]),
       "amount": pick(ddt, T_CPTY_DIM, ["amount"]),
       "n_txn": pick(ddt, T_CPTY_DIM, ["n_txn"]),
       "first_month": pick(ddt, T_CPTY_DIM, ["first_month"], required=False),
       "last_month": pick(ddt, T_CPTY_DIM, ["last_month"], required=False),
       "is_payee": pick(ddt, T_CPTY_DIM, ["is_payee"], required=False),
       "is_payer": pick(ddt, T_CPTY_DIM, ["is_payer"], required=False)}
CUSTC = {"master": pick(udt, T_CUST_DIM, ["master_pwr_id"], required=False),
         "party_type": pick(udt, T_CUST_DIM, ["party_type"], required=False),
         "naics2": pick(udt, T_CUST_DIM, ["naics2"], required=False),
         "naics_status": pick(udt, T_CUST_DIM, ["naics_status"], required=False),
         "name": pick(udt, T_CUST_DIM, ["customer_name"], required=False)}

ALL_MONTHS = month_range(HISTORY_START, LAST_MONTH)
STATUS = {m: month_status(m) for m in ALL_MONTHS}
if REBUILD_FROM:
    assert REBUILD_FROM in ALL_MONTHS, f"REBUILD_FROM {REBUILD_FROM} outside {HISTORY_START} → {LAST_MONTH}"
    start = REBUILD_FROM
else:
    start = next((m for m in ALL_MONTHS if STATUS[m] != "done"), None)
PLAN = [m for m in ALL_MONTHS[ALL_MONTHS.index(start):] if REBUILD_FROM or STATUS[m] != "done"] if start else []
if MAX_MONTHS_PER_RUN:
    PLAN = PLAN[:MAX_MONTHS_PER_RUN]

# inputs for every month to build
need_parts = {T_EDGE_M: partitions(T_EDGE_M), T_CPTY_MONTH: partitions(T_CPTY_MONTH)}
if USE_DAILY:
    need_parts[T_EDGE_D] = partitions(T_EDGE_D)
gaps = {t: [m for m in PLAN if m not in p] for t, p in need_parts.items()}
gaps = {t: g for t, g in gaps.items() if g}
assert not gaps, f"input partitions missing for months to build: {gaps}"

# the first month to build needs the previous month's churn state (same configuration)
if PLAN and PLAN[0] != HISTORY_START and not REBUILD_FROM:
    pm = prev_month(PLAN[0])
    for kind in ["rel", "fi"]:
        if not exists(state_path(kind, pm) + "/_SUCCESS"):
            raise RuntimeError(f"churn state for {pm} ({kind}) is missing (pruned or another configuration). "
                               f"Set REBUILD_FROM = '{HISTORY_START}' to rebuild the history.")
if REBUILD_FROM and REBUILD_FROM != HISTORY_START:
    pm = prev_month(REBUILD_FROM)
    if not all(exists(state_path(k, pm) + "/_SUCCESS") for k in ["rel", "fi"]):
        raise RuntimeError(f"REBUILD_FROM = {REBUILD_FROM} needs the churn state of {pm}, which is no longer kept "
                           f"(KEEP_STATE = {KEEP_STATE}). Use REBUILD_FROM = '{HISTORY_START}'.")

plan_set = set(PLAN)
rows = []
for m in ALL_MONTHS:
    action = ("rebuild" if STATUS[m] == "done" else "build") if m in plan_set else (
             "keep" if STATUS[m] == "done" else "later run (MAX_MONTHS_PER_RUN)")
    rows.append({"month": m, "status": STATUS[m], "action": action})
plan_df = pd.DataFrame(rows)
cnt = plan_df.groupby(["status", "action"]).size().reset_index(name="months")
other_sigs = sorted(set(ls_dirs(P_STATE_ROOT) + ls_dirs(P_WORK_ROOT)) - {SIG})
show(f"§1 Plan — {len(PLAN)} month(s) to build this run" + (f": {PLAN[0]} → {PLAN[-1]}" if PLAN else ""),
     ("Summary", cnt),
     ("Months", plan_df if len(plan_df) <= 40 else pd.concat([plan_df.head(20), plan_df.tail(20)])),
     note(f"Canonical FI tables: {'found' if HAVE_CANON else 'NOT found — raw FI names used'}. "
          f"Daily active days: {USE_DAILY}. Signature {SIG} (map files {FP['map']}; FI files {FP['fi']})."),
     note("stale = built under another signature: a setting, the entity map or the canonical FI tables changed "
          "since (rerunning pkg_fi_canonical or the map notebook rewrites the table and counts as a change).")
     if "stale" in STATUS.values() else None,
     note(f"Other configuration folders on disk: {other_sigs or 'none'} (see §6)."),
     note("REBUILD_FROM is set: those months' outputs, state and scratch are deleted when §3 starts." if REBUILD_FROM else
          "Nothing is deleted; months already done are kept."))

## §1b Entity map: one row per id

The map is copied once per signature into the scratch folder with **one row per `cpty_id`** (an id listed twice would
be counted twice in every month; the exact reconciliation in §3 stops the month when that happens). When an id has
several rows, the row with the lowest `ent_l2` is kept. The check below counts duplicates and shows examples with their
row counts in the v1 map and in `pkg_cpty_dim`, to fix them at the source.

In [ ]:
# §1b  Deduplicated entity map (built once per signature) and duplicate check
P_MAP_U, P_MAP_QA, P_MAP_EX = (f"{P_WORK}/_map_one_row_per_id", f"{P_WORK}/_map_qa", f"{P_WORK}/_map_dup_examples")
t0 = time.time()
map_built = not all(exists(x + "/_SUCCESS") for x in (P_MAP_U, P_MAP_QA, P_MAP_EX))
if map_built:
    (spark.table(T_MAP).groupBy("cpty_id")
          .agg(F.min(F.struct("ent_l2", "name_class", "type_reason", "cpty_id_src")).alias("r"),
               F.max("ent_l2").alias("ent_max"), F.count("*").alias("map_rows"))
          .select("cpty_id", F.col("r.ent_l2").alias("ent"), F.col("r.name_class").alias("name_class"),
                  F.col("r.type_reason").alias("type_reason"), F.col("r.cpty_id_src").alias("id_src"), "map_rows",
                  (F.col("r.ent_l2") != F.col("ent_max")).alias("ent_conflict"))
          .write.mode("overwrite").parquet(P_MAP_U))
    mu = spark.read.parquet(P_MAP_U)
    q = mu.agg(F.count("*").alias("ids"), F.sum((F.col("map_rows") > 1).cast("long")).alias("ids_with_2plus_rows"),
               F.sum(F.col("map_rows") - 1).alias("extra_rows"),
               F.sum(F.col("ent_conflict").cast("long")).alias("ids_with_2plus_entities")).first().asDict()
    q["cpty_dim_ids_with_2plus_rows"] = spark.table(T_CPTY_DIM).groupBy("cpty_id").count().where("count > 1").count()
    spark.createDataFrame(pd.DataFrame([{k: int(v or 0) for k, v in q.items()}])).write.mode("overwrite").parquet(P_MAP_QA)
    ex_ids = [r[0] for r in mu.where("map_rows > 1").orderBy(F.desc("map_rows")).select("cpty_id").limit(10).collect()]
    ex = (spark.table(T_MAP).where(F.col("cpty_id").isin(ex_ids))
               .select("cpty_id", "cpty_id_src", "ent_l1", "ent_l2", "rule", "name_class", "type_reason", "name_clean"))
    for t, c in [(T_MAP_V1, "rows_in_map_v1"), (T_CPTY_DIM, "rows_in_cpty_dim")]:
        if table_exists(t):
            ex = ex.join(spark.table(t).where(F.col("cpty_id").isin(ex_ids)).groupBy("cpty_id")
                              .agg(F.count("*").alias(c)), "cpty_id", "left")
    ex.write.mode("overwrite").parquet(P_MAP_EX)
MAP_U = spark.read.parquet(P_MAP_U)
qa = spark.read.parquet(P_MAP_QA).toPandas().T.reset_index()
qa.columns = ["check", "value"]
ex = spark.read.parquet(P_MAP_EX).toPandas()
dups = int(qa.loc[qa.check == "ids_with_2plus_rows", "value"].iloc[0])
show(f"§1b Entity map, one row per id — {'built' if map_built else 'reused'} ({(time.time() - t0) / 60:.1f} min)",
     ("Duplicate check", qa),
     ("Examples of ids with several map rows", ex.sort_values(["cpty_id", "ent_l2"])) if len(ex) else None,
     note(f"{dups:,} ids had more than one row in {T_MAP}; the lowest ent_l2 is kept for each. Fix in the map "
          "notebook when convenient (the rows_in_* columns show where the duplication starts)." if dups else
          "No duplicate ids in the map."))

## §2 Entity dimension — `pkg_cpty_entity_dim`

One row per entity over all history: dominant name / class / type (by lifetime dollars), number of ids, holding banks
(canonical), first and last month, lifetime flow, payer / payee flags, and a match of its name to PNC **organisation**
customers (`pnc_match_n`, `pnc_tm_match_n`; business / unknown / bank classes, legal words removed, exact match).
Rebuilt only when the map or the canonical FI tables change, or with `FORCE_ENT_DIM`.

In [ ]:
# §2  Build or reuse the entity dimension
DIM_SIG = hashlib.md5(json.dumps({"map": T_MAP, "fp": FP, "dim": table_fp(T_CPTY_DIM), "cust": table_fp(T_CUST_DIM),
                                  "v": METRICS_VERSION, "dv": ENT_DIM_VERSION}, sort_keys=True).encode()).hexdigest()[:10]
DIM_MARK = f"{P_ENT_DIM}/_pkg_sig_{DIM_SIG}"
t0 = time.time()
dim_built = FORCE_ENT_DIM or not (exists(P_ENT_DIM + "/_SUCCESS") and exists(DIM_MARK))
if dim_built:
    lit_null = lambda typ: F.lit(None).cast(typ)
    dsel = spark.table(T_CPTY_DIM).select(
        "cpty_id", F.col(DIM["amount"]).cast(DEC).alias("amount"), F.col(DIM["name"]).alias("cpty_name"),
        F.col(DIM["fi"]).alias("fi"), F.col(DIM["cpty_type"]).alias("cpty_type"),
        F.col(DIM["n_txn"]).cast("long").alias("n_txn"),
        (F.col(DIM["first_month"]) if DIM["first_month"] else lit_null("string")).alias("first_month"),
        (F.col(DIM["last_month"]) if DIM["last_month"] else lit_null("string")).alias("last_month"),
        (F.col(DIM["is_payee"]).cast("boolean") if DIM["is_payee"] else lit_null("boolean")).alias("is_payee"),
        (F.col(DIM["is_payer"]).cast("boolean") if DIM["is_payer"] else lit_null("boolean")).alias("is_payer"))
    dcols = [c for c in dsel.columns if c != "cpty_id"]
    dim_u = (dsel.groupBy("cpty_id").agg(F.max(F.struct(*dcols)).alias("d"))      # one row per id (largest amount)
                 .select("cpty_id", *[F.col(f"d.{c}").alias(c) for c in dcols]))
    ids = (MAP_U.select("cpty_id", "ent", "name_class", "type_reason", "id_src")
                .join(dim_u, "cpty_id", "left"))
    ids = canon_fi(ids, "cpty_id", "fi")
    g = ids.groupBy("ent").agg(
        F.count("*").alias("n_ids"),
        F.max(F.struct(F.coalesce("amount", F.lit(0).cast(DEC)).alias("a"), "cpty_id", "cpty_name", "name_class",
                       "type_reason", "cpty_type", "id_src")).alias("dom"),          # ties → highest cpty_id
        F.sum("amount").cast(DEC).alias("amt_lifetime"), F.sum("n_txn").alias("n_txn_lifetime"),
        F.min("first_month").alias("first_month"), F.max("last_month").alias("last_month"),
        F.max(F.col("is_payee").cast("int")).cast("boolean").alias("is_payee"),
        F.max(F.col("is_payer").cast("int")).cast("boolean").alias("is_payer"),
        F.countDistinct("fi_c").alias("n_fi"))
    fid = (ids.where(F.col("fi_c").isNotNull()).groupBy("ent", "fi_c").agg(F.sum("amount").alias("a"))
              .groupBy("ent").agg(F.max(F.struct("a", "fi_c")).getField("fi_c").alias("fi_dom")))
    dim0 = (g.join(fid, "ent", "left")
             .select(F.col("ent").alias("ent_id"), F.col("dom.cpty_name").alias("ent_name"),
                     F.col("dom.name_class").alias("name_class"), F.col("dom.type_reason").alias("type_reason"),
                     F.col("dom.cpty_type").alias("cpty_type"), F.col("dom.id_src").alias("id_src"),
                     "n_ids", "n_fi", "fi_dom", "first_month", "last_month", "amt_lifetime", "n_txn_lifetime",
                     "is_payee", "is_payer"))
    p_dim0 = f"{P_WORK}/_stage_ent_dim0"            # written once: the split below must see one fixed version
    dim0.write.mode("overwrite").parquet(p_dim0)
    dimx = spark.read.parquet(p_dim0)
    # name match to PNC organisation customers
    if CUSTC["name"]:
        cu = spark.table(T_CUST_DIM)
        if CUSTC["party_type"]:
            cu = cu.where(F.col(CUSTC["party_type"]) == "O")
        ck = (cu.select(simple_key(F.col(CUSTC["name"])).alias("k"), F.col("is_tm").cast("boolean").alias("tm"))
                .where(F.length("k") >= 4)
                .groupBy("k").agg(F.count("*").alias("pnc_match_n"),
                                  F.sum(F.col("tm").cast("int")).alias("pnc_tm_match_n")))
        # only rows with a key go through the join (NULL keys would all land in one task)
        dimx = dimx.withColumn("k", F.when(F.col("name_class").isin("business", "unknown", "bank"),
                                           simple_key(F.col("ent_name"))))
        dimx = dimx.withColumn("k", F.when(F.length("k") >= 4, F.col("k")))
        zero = F.lit(0).cast("long")
        dimx = (dimx.where(F.col("k").isNotNull()).join(ck, "k", "left")
                    .unionByName(dimx.where(F.col("k").isNull())
                                     .withColumn("pnc_match_n", zero).withColumn("pnc_tm_match_n", zero))
                    .drop("k")
                    .fillna(0, ["pnc_match_n", "pnc_tm_match_n"]))
    else:
        dimx = dimx.withColumn("pnc_match_n", F.lit(0).cast("long")).withColumn("pnc_tm_match_n", F.lit(0).cast("long"))
    stage = f"{P_WORK}/_stage_ent_dim"
    dimx.write.mode("overwrite").parquet(stage)
    chk = spark.read.parquet(stage).agg(F.count("*").alias("n"), F.countDistinct("ent_id").alias("u")).first()
    assert chk["n"] == chk["u"], f"pkg_cpty_entity_dim: {chk['n'] - chk['u']:,} duplicate ent_id rows"
    touch(f"{stage}/_pkg_sig_{DIM_SIG}")
    delete(P_ENT_DIM)
    rename(stage, P_ENT_DIM)
    spark.sql(f"DROP TABLE IF EXISTS {T_ENT_DIM}")
    cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in spark.read.parquet(P_ENT_DIM).schema.fields)
    spark.sql(f"CREATE EXTERNAL TABLE {T_ENT_DIM} ({cols}) STORED AS PARQUET LOCATION '{P_ENT_DIM}'")

ENT_DIM = spark.read.parquet(P_ENT_DIM)
dsum = (ENT_DIM.groupBy("name_class")
               .agg(F.count("*").alias("entities"), F.sum("n_ids").alias("ids"),
                    F.sum(F.col("amt_lifetime").cast("double")).alias("amt"),
                    F.sum((F.col("pnc_match_n") > 0).cast("int")).alias("match a PNC org customer"))
               .toPandas().sort_values("amt", ascending=False))
dsum["$B lifetime"] = dsum.amt / 1e9
show(f"§2 Entity dimension — {'built' if dim_built else 'reused'} ({(time.time() - t0) / 60:.1f} min) → {T_ENT_DIM}",
     ("By name_class", dsum[["name_class", "entities", "ids", "$B lifetime", "match a PNC org customer"]]),
     note(f"Impala: INVALIDATE METADATA {T_ENT_DIM};" if dim_built else "Unchanged since the last build."))

## §3 Monthly build

Steps per month (each written to disk before the next; a rerun reuses finished steps):

| Step | Output | Where |
|---|---|---|
| `idrel` | id × customer × direction: dollars, transactions, self dollars, dollars by rail, entity | scratch |
| `rel` | entity × customer × direction | scratch |
| `days` | entity × direction: active days (if `USE_DAILY`) | scratch |
| `relstate` | churn state: entity × direction × customer, 12-month presence mask, last dollars, class this month | state |
| `fistate` | bank state: entity × canonical bank, presence mask, dollars this month | state |
| `churn` | entity × direction churn counts and dollars | scratch |
| `slim` | entity: flow, customers, ids | scratch |
| `elig` | entity: trailing-window dollars, months active, eligible | scratch |
| `full` | entity: full metric set, eligible entities only | scratch |
| publish | assemble → staging → exact reconciliation → rename into `month=…` → register partition | output |

After a month is published: run log row, state older than `KEEP_STATE` months deleted, scratch deleted.

In [ ]:
# §3a  Step functions
D0 = F.lit(0).cast(DEC)
MAP_DF = MAP_U.select("cpty_id", "ent")                                  # one row per id (§1b)
CUST = spark.table(T_CUST_DIM).select(
    F.col("mdm_id").alias("cust"),
    F.coalesce(F.col("is_tm").cast("boolean"), F.lit(False)).alias("is_tm"),
    (F.col(CUSTC["master"]) if CUSTC["master"] else F.lit(None).cast("string")).alias("master"),
    ((F.col(CUSTC["party_type"]) == "O") if CUSTC["party_type"] else F.lit(False)).alias("is_org"),
    (F.when(F.col(CUSTC["naics_status"]) == "valid", F.col(CUSTC["naics2"]))
     if CUSTC["naics2"] and CUSTC["naics_status"] else F.lit(None).cast("string")).alias("naics2"),
    (F.col(CUSTC["name"]) if CUSTC["name"] else F.lit(None).cast("string")).alias("cust_name"))
LABELS = (spark.read.parquet(P_ENT_DIM).select(F.col("ent_id").alias("ent"), "ent_name", "name_class", "cpty_type", "id_src")
               .dropDuplicates(["ent"]))
STEP_LOG = []

def run_step(m, name, fn, where="work"):
    # Run fn() → DataFrame, write it, mark done; reuse when already done. where = "work" | "rel" | "fi" (state)
    path = work_path(m, name) if where == "work" else state_path(where, m)
    t0 = time.time()
    if exists(path + "/_SUCCESS"):
        STEP_LOG.append({"month": m, "step": name, "action": "reused", "min": 0.0})
        return spark.read.parquet(path)
    df = fn()
    df.write.mode("overwrite").parquet(path)
    STEP_LOG.append({"month": m, "step": name, "action": "built", "min": (time.time() - t0) / 60})
    return spark.read.parquet(path)

def edges_cpty(m):
    to_c = F.col("dst_kind") == "CPTY"
    return (spark.table(T_EDGE_M).where(F.col("month") == m)
                 .where((F.col("src_kind") == "CPTY") | to_c)
                 .select(F.when(to_c, F.col("dst_id")).otherwise(F.col("src_id")).alias("cpty_id"),
                         F.when(to_c, F.col("src_id")).otherwise(F.col("dst_id")).alias("cust"),
                         F.when(to_c, F.lit("in")).otherwise(F.lit("out")).alias("dir"),
                         F.substring("category", 1, 1).alias("rnum"),
                         F.coalesce(F.col("is_self"), F.lit(False)).alias("is_self"),
                         F.col("amount").cast(DEC).alias("amount"), F.col("n_txn").cast("long").alias("n_txn")))

def s_idrel(m):
    e = edges_cpty(m)
    agg = [F.sum("amount").cast(DEC).alias("amt"), F.sum("n_txn").alias("n_txn"),
           F.sum(F.when(F.col("is_self"), F.col("amount")).otherwise(D0)).cast(DEC).alias("amt_self")]
    agg += [F.sum(F.when(F.col("rnum").isin(RAIL_NUMS[r]), F.col("amount")).otherwise(D0)).cast(DEC).alias(f"amt_{r}")
            for r in RAILS]
    return (e.groupBy("cpty_id", "cust", "dir").agg(*agg)
             .join(MAP_DF, "cpty_id", "left")
             .withColumn("in_map", F.col("ent").isNotNull())
             .withColumn("ent", F.coalesce("ent", "cpty_id")))

def s_rel(idrel):
    sums = ["amt", "amt_self"] + [f"amt_{r}" for r in RAILS]
    return idrel.groupBy("ent", "cust", "dir").agg(*[F.sum(c).cast(DEC).alias(c) for c in sums],
                                                   F.sum("n_txn").alias("n_txn"))

def s_days(m, idrel):
    to_c = F.col("dst_kind") == "CPTY"
    d = (spark.table(T_EDGE_D).where(F.col("month") == m).where((F.col("src_kind") == "CPTY") | to_c)
              .select(F.when(to_c, F.col("dst_id")).otherwise(F.col("src_id")).alias("cpty_id"),
                      F.when(to_c, F.lit("in")).otherwise(F.lit("out")).alias("dir"), "txn_date").distinct())
    idmap = idrel.select("cpty_id", "ent").distinct()        # this month's ids only (ids absent here have no days)
    return (d.join(idmap, "cpty_id", "inner")
             .select("ent", "dir", "txn_date").distinct()
             .groupBy("ent", "dir").agg(F.count("*").alias("active_days")))

def _prev_state(kind, m, cols_types):
    if m == HISTORY_START:
        return spark.createDataFrame([], ", ".join(f"{c} {t}" for c, t in cols_types))
    p = state_path(kind, prev_month(m))
    if not exists(p + "/_SUCCESS"):
        raise RuntimeError(f"churn state {kind} for {prev_month(m)} missing; set REBUILD_FROM = '{HISTORY_START}'")
    return spark.read.parquet(p).select(*[c for c, _ in cols_types])

FULL_MASK = (1 << LOOKBACK) - 1
REC_MASK = (1 << REC_WIN) - 1

def s_relstate(m, rel):
    prev = _prev_state("rel", m, [("ent", "string"), ("dir", "string"), ("cust", "string"), ("mask", "int"),
                                  ("last_amt", DEC)])
    cur = rel.select("ent", "dir", "cust", F.col("amt").alias("amt_now"))
    j = (prev.join(cur, ["ent", "dir", "cust"], "full")
             .withColumn("pm", F.coalesce(F.col("mask"), F.lit(0)))
             .withColumn("present", F.col("amt_now").isNotNull())
             .withColumn("was_last", F.expr("(pm & 1) = 1"))
             .withColumn("rec_std", F.expr(f"bit_count(pm & {REC_MASK}) >= {REC_MIN}"))
             .withColumn("new_mask", F.expr(f"(shiftleft(pm, 1) | cast(present as int)) & {FULL_MASK}")))
    cls = (F.when(F.col("present") & F.col("was_last"), "retained")
            .when(F.col("present") & (F.col("pm") != 0), "reactivated")
            .when(F.col("present"), "new")
            .when(F.col("was_last"), "lost"))
    return (j.where(F.col("new_mask") != 0)
             .select("ent", "dir", "cust", F.col("new_mask").alias("mask"),
                     F.when(F.col("present"), F.col("amt_now")).otherwise(F.col("last_amt")).cast(DEC).alias("last_amt"),
                     "amt_now", F.col("last_amt").alias("prev_last_amt"), cls.alias("cls"), "was_last", "rec_std",
                     (F.col("rec_std") & ~F.col("present")).alias("rec_brk")))

def s_fistate(m, idrel):
    idm = idrel.groupBy("cpty_id", "ent").agg(F.sum("amt").cast(DEC).alias("amt"))
    fim = (spark.table(T_CPTY_MONTH).where((F.col("month") == m) & F.col("fi").isNotNull())
                .groupBy("cpty_id").agg(F.expr("max_by(fi, amount)").alias("fi")))
    x = canon_fi(idm.join(fim, "cpty_id", "left"), "cpty_id", "fi")
    cur = x.where(F.col("fi_c").isNotNull()).groupBy("ent", "fi_c").agg(F.sum("amt").cast(DEC).alias("amt_now"))
    prev = _prev_state("fi", m, [("ent", "string"), ("fi_c", "string"), ("mask", "int")])
    j = (prev.join(cur, ["ent", "fi_c"], "full")
             .withColumn("pm", F.coalesce(F.col("mask"), F.lit(0)))
             .withColumn("present", F.col("amt_now").isNotNull())
             .withColumn("was_last", F.expr("(pm & 1) = 1"))
             .withColumn("new_mask", F.expr(f"(shiftleft(pm, 1) | cast(present as int)) & {FULL_MASK}")))
    return (j.where(F.col("new_mask") != 0)
             .select("ent", "fi_c", F.col("new_mask").alias("mask"), "amt_now",
                     (F.col("present") & (F.col("pm") == 0)).alias("fi_new"),
                     (F.col("was_last") & ~F.col("present")).alias("fi_lost")))

def s_churn(relstate):
    c = F.col("cls")
    return relstate.groupBy("ent", "dir").agg(
        *[F.sum((c == k).cast("long")).alias(f"{k}_cust") for k in ["new", "reactivated", "retained", "lost"]],
        F.sum(F.when(c == "new", F.col("amt_now"))).cast("double").alias("new_amt"),
        F.sum(F.when(c == "lost", F.col("prev_last_amt"))).cast("double").alias("lost_amt"),
        F.sum(F.when(F.col("was_last"), F.col("prev_last_amt"))).cast("double").alias("prev_amt"),
        F.sum(F.col("rec_std").cast("long")).alias("rec_standing"),
        F.sum(F.col("rec_brk").cast("long")).alias("rec_broken"),
        F.sum(F.when(F.col("rec_std"), F.col("prev_last_amt"))).cast("double").alias("rec_std_amt"),
        F.sum(F.when(F.col("rec_brk"), F.col("prev_last_amt"))).cast("double").alias("rec_brk_amt"))

def pivot_dirs(df, cols):
    return df.groupBy("ent").agg(*[F.max(F.when(F.col("dir") == d, F.col(c))).alias(f"{c}_{d}")
                                   for c in cols for d in DIRS])

def s_slim(rel, idrel):
    s = rel.groupBy("ent", "dir").agg(F.sum("amt").cast(DEC).alias("amt"), F.sum("n_txn").alias("n_txn"),
                                      F.count("*").alias("n_cust"))
    p = pivot_dirs(s, ["amt", "n_txn", "n_cust"])
    nall = rel.select("ent", "cust").distinct().groupBy("ent").agg(F.count("*").alias("n_cust_all"))
    nids = idrel.select("ent", "cpty_id").distinct().groupBy("ent").agg(F.count("*").alias("n_ids_active"))
    out = p.join(nall, "ent", "left").join(nids, "ent", "left")
    for d in DIRS:
        out = (out.withColumn(f"amt_{d}", F.coalesce(F.col(f"amt_{d}"), D0))
                  .withColumn(f"n_txn_{d}", F.coalesce(F.col(f"n_txn_{d}"), F.lit(0).cast("long")))
                  .withColumn(f"n_cust_{d}", F.coalesce(F.col(f"n_cust_{d}"), F.lit(0).cast("long"))))
    return out.withColumn("amt_all", (F.col("amt_in") + F.col("amt_out")).cast(DEC))

def s_elig(m, slim):
    prior = [month_add(m, -k) for k in range(1, ELIG_WINDOW) if month_add(m, -k) >= HISTORY_START]
    miss = [x for x in prior if month_status(x) != "done"]
    if miss:
        raise RuntimeError(f"eligibility for {m} needs months {miss}, which are not built with this configuration")
    u = slim.select("ent", "amt_all")
    if prior:
        prev = (spark.read.option("basePath", P_OUT).parquet(*[out_path(x) for x in prior])
                     .select(F.col("ent_id").alias("ent"), F.col("amt_all").cast(DEC).alias("amt_all")))
        u = u.unionByName(prev)
    hist = min(ELIG_WINDOW, len(month_range(HISTORY_START, m)))
    return (u.groupBy("ent").agg(F.sum("amt_all").cast(DEC).alias("amt_12m"),
                                 F.sum((F.col("amt_all") > 0).cast("long")).alias("active_months_12m"))
             .withColumn("eligible", F.col("amt_12m") >= F.lit(ELIG_AMT))
             .withColumn("history_months", F.lit(hist).cast("int")))

NAICS2_MAP = F.create_map(*[F.lit(x) for kv in NAICS2_DESC.items() for x in kv])
DIR_FULL = (["avg_ticket", "active_days", "amt_self", "n_cust_tm", "n_family", "tm_share", "org_share", "top1_share",
             "hhi", "top_cust", "top_cust_name", "n_naics2", "naics2_top", "naics2_top_desc", "naics2_top_share",
             "naics2_entropy", "dep_wavg",
             "n_cust_dep50", "new_cust", "reactivated_cust", "retained_cust", "lost_cust", "retention",
             "new_amt_share", "lost_amt_share", "rec_standing", "rec_broken", "rec_broken_amt_share"]
            + [f"share_{r}" for r in RAILS] + ["n_rails"])

def s_full(rel, elig, churn, days):
    el = elig.where("eligible").select("ent")
    base = churn.join(el, "ent", "left_semi")
    ct = rel.groupBy("cust", "dir").agg(F.sum(F.col("amt").cast("double")).alias("cust_tot"))
    r = (rel.join(el, "ent", "left_semi").join(CUST, "cust", "left").join(ct, ["cust", "dir"], "left")
            .withColumn("a", F.col("amt").cast("double"))
            .withColumn("dep", F.col("a") / F.col("cust_tot")))
    a = r.groupBy("ent", "dir").agg(
        F.sum("a").alias("amt_d"), F.sum("n_txn").alias("n_txn"),
        F.sum(F.col("is_tm").cast("long")).alias("n_cust_tm"),
        F.sum(F.when(F.col("is_tm"), F.col("a"))).alias("amt_tm"),
        F.sum(F.when(F.col("is_org"), F.col("a"))).alias("amt_org"),
        F.sum("amt_self").cast(DEC).alias("amt_self"),
        *[F.sum(F.col(f"amt_{x}").cast("double")).alias(f"amt_{x}") for x in RAILS],
        F.sum(F.col("a") * F.col("a")).alias("sq"),
        F.expr("max_by(struct(cust, cust_name), a)").alias("top"), F.max("a").alias("top_amt"),
        F.sum(F.col("a") * F.col("dep")).alias("dep_num"),
        F.sum((F.col("dep") >= DEP_MIN).cast("long")).alias("n_cust_dep50"),
        F.sum(F.col("naics2").isNotNull().cast("long")).alias("n_naics_valid"))
    fam = (r.select("ent", "dir", F.coalesce("master", "cust").alias("fam")).distinct()
            .groupBy("ent", "dir").agg(F.count("*").alias("n_family")))
    sec = (r.where(F.col("naics2").isNotNull() & (F.col("a") > 0))
            .groupBy("ent", "dir", "naics2").agg(F.sum("a").alias("s"))
            .groupBy("ent", "dir").agg(F.count("*").alias("n_naics2_raw"), F.expr("max_by(naics2, s)").alias("ntop"),
                                       F.max("s").alias("top_s"), F.sum("s").alias("S"),
                                       F.sum(F.col("s") * F.log(F.col("s"))).alias("slog")))
    per = base.join(a, ["ent", "dir"], "left").join(fam, ["ent", "dir"], "left").join(sec, ["ent", "dir"], "left")
    if days is not None:
        per = per.join(days, ["ent", "dir"], "left")
    else:
        per = per.withColumn("active_days", F.lit(None).cast("long"))
    ok_naics = F.coalesce(F.col("n_naics_valid"), F.lit(0)) >= NAICS_MIN
    amt = F.col("amt_d")
    per = (per
        .withColumn("avg_ticket", amt / F.col("n_txn"))
        .withColumn("active_days", F.coalesce(F.col("active_days"), F.lit(0).cast("long")) if days is not None
                    else F.col("active_days"))
        .withColumn("tm_share", F.coalesce(F.col("amt_tm"), F.lit(0.0)) / amt)
        .withColumn("org_share", F.coalesce(F.col("amt_org"), F.lit(0.0)) / amt)
        .withColumn("top1_share", F.col("top_amt") / amt)
        .withColumn("hhi", F.col("sq") / (amt * amt))
        .withColumn("dep_wavg", F.col("dep_num") / amt)
        .withColumn("n_naics2", F.when(ok_naics, F.coalesce(F.col("n_naics2_raw"), F.lit(0)).cast("long")))
        .withColumn("naics2_top", F.when(ok_naics, F.col("ntop")))
        .withColumn("naics2_top_desc", F.when(ok_naics, NAICS2_MAP[F.col("ntop")]))
        .withColumn("top_cust", F.col("top.cust"))
        .withColumn("top_cust_name", F.col("top.cust_name"))
        .withColumn("naics2_top_share", F.when(ok_naics, F.col("top_s") / F.col("S")))
        .withColumn("naics2_entropy", F.when(ok_naics, F.log(F.col("S")) - F.col("slog") / F.col("S")))
        .withColumn("retention", F.col("retained_cust") / (F.col("retained_cust") + F.col("lost_cust")))
        .withColumn("new_amt_share", F.coalesce(F.col("new_amt"), F.lit(0.0)) / amt)
        .withColumn("lost_amt_share", F.coalesce(F.col("lost_amt"), F.lit(0.0)) / F.col("prev_amt"))
        .withColumn("rec_broken_amt_share", F.coalesce(F.col("rec_brk_amt"), F.lit(0.0)) / F.col("rec_std_amt"))
        .withColumn("n_rails", sum([F.when(F.col(f"amt_{x}") > 0, 1).otherwise(0) for x in RAILS]).cast("long")))
    for x in RAILS:
        per = per.withColumn(f"share_{x}", F.col(f"amt_{x}") / amt)
    for c in ["n_cust_tm", "n_family", "n_cust_dep50"]:
        per = per.withColumn(c, F.coalesce(F.col(c), F.lit(0).cast("long")))
    per = per.withColumn("amt_self", F.coalesce(F.col("amt_self"), D0))
    return pivot_dirs(per, DIR_FULL)

ZERO_COLS = [f"{c}_{d}" for c in ["amt_self", "n_cust_tm", "n_family", "n_cust_dep50", "new_cust", "reactivated_cust",
                                   "retained_cust", "lost_cust", "rec_standing", "rec_broken", "n_rails"]
             + (["active_days"] if USE_DAILY else []) for d in DIRS]

def reconcile(m, df):
    src = edges_cpty(m).agg(F.sum("amount").alias("amt"), F.sum("n_txn").alias("n_txn")).first()
    def culprits():
        dup = df.groupBy("ent_id").count().where("count > 1").limit(10).toPandas()
        if len(dup):
            return f" duplicate ent_id: {dup.to_dict('records')}"
        rel = spark.read.parquet(work_path(m, "rel")).groupBy("ent").agg(F.sum("amt").alias("edge_amt"))
        diff = (df.select(F.col("ent_id").alias("ent"), "amt_all").join(rel, "ent", "full")
                  .where(~F.col("amt_all").eqNullSafe(F.col("edge_amt"))).limit(10).toPandas())
        return f" entities whose $ differ: {diff.to_dict('records')}"
    got = df.agg(F.sum("amt_all").alias("amt"), F.sum(F.col("n_txn_in") + F.col("n_txn_out")).alias("n_txn"),
                 F.count("*").alias("rows"), F.countDistinct("ent_id").alias("ents"),
                 F.sum(F.col("eligible").cast("long")).alias("elig"),
                 F.sum(F.when(F.col("eligible"), F.col("amt_all"))).alias("amt_elig")).first()
    if not ((src["amt"] or 0) == (got["amt"] or 0) and (src["n_txn"] or 0) == (got["n_txn"] or 0)):
        raise AssertionError(f"{m}: output $ {got['amt']} / txn {got['n_txn']} ≠ edges $ {src['amt']} / "
                             f"txn {src['n_txn']}." + culprits())
    assert got["rows"] == got["ents"], f"{m}: duplicate ent_id rows"
    return got

def sync_partitions():
    # register every month folder that is complete; drop registrations whose folder is gone
    if not table_exists(T_OUT):
        return
    reg = partitions(T_OUT)
    for x in sorted(reg | set(ALL_MONTHS)):
        have = exists(out_path(x) + "/_SUCCESS")
        if have and x not in reg:
            spark.sql(f"ALTER TABLE {T_OUT} ADD IF NOT EXISTS PARTITION (month='{x}') LOCATION '{out_path(x)}'")
        elif not have and x in reg:
            spark.sql(f"ALTER TABLE {T_OUT} DROP IF EXISTS PARTITION (month='{x}')")

def ensure_table(m):
    want = [(f.name, f.dataType.simpleString()) for f in spark.read.parquet(out_path(m)).schema.fields]
    if table_exists(T_OUT) and spark.table(T_OUT).dtypes != want + [("month", "string")]:
        spark.sql(f"DROP TABLE {T_OUT}")                       # external table: drops the definition, not the files
    if not table_exists(T_OUT):
        cols = ", ".join(f"`{c}` {t}" for c, t in want)
        spark.sql(f"CREATE EXTERNAL TABLE {T_OUT} ({cols}) PARTITIONED BY (month STRING) "
                  f"STORED AS PARQUET LOCATION '{P_OUT}'")
    sync_partitions()

def prune_state(m):
    keep_from = month_add(m, -(KEEP_STATE - 1))
    for kind in ["rel", "fi"]:
        for d_ in ls_dirs(f"{P_STATE}/{kind}"):
            if d_.startswith("month=") and d_.split("=", 1)[1] < keep_from:
                delete(f"{P_STATE}/{kind}/{d_}")

def build_month(m, on_step):
    on_step(m, "idrel")
    idrel = run_step(m, "idrel", lambda: s_idrel(m))
    on_step(m, "rel")
    rel = run_step(m, "rel", lambda: s_rel(idrel))
    days = None
    if USE_DAILY:
        on_step(m, "days")
        days = run_step(m, "days", lambda: s_days(m, idrel))
    on_step(m, "relstate")
    relstate = run_step(m, "relstate", lambda: s_relstate(m, rel), where="rel")
    on_step(m, "fistate")
    fistate = run_step(m, "fistate", lambda: s_fistate(m, idrel), where="fi")
    on_step(m, "churn")
    churn = run_step(m, "churn", lambda: s_churn(relstate))
    on_step(m, "slim")
    slim = run_step(m, "slim", lambda: s_slim(rel, idrel))
    on_step(m, "elig")
    elig = run_step(m, "elig", lambda: s_elig(m, slim))
    on_step(m, "full")
    full = run_step(m, "full", lambda: s_full(rel, elig, churn, days))

    on_step(m, "publish")
    # hub cut-offs among eligible entities
    hub_in = slim.join(elig.where("eligible").select("ent"), "ent", "left_semi").select(
        F.col("n_cust_all").cast("double").alias("deg"), F.col("amt_all").cast("double").alias("str"))
    # Tail cut-offs are exact: the value of the k-th largest entity, k = n − ceil(q·n) + 1. (approxQuantile's rank
    # error, 1e-4 × n, is as wide as the whole P99.99 tail and returned values near the maximum.)
    hub_in = hub_in.cache()
    n_h = hub_in.count()
    def cuts(col):
        if n_h == 0:
            return []
        ks = [n_h - math.ceil(q * n_h) + 1 for q in HUB_Q]
        top = [r[0] for r in hub_in.select(col).orderBy(F.desc(col)).limit(max(ks)).collect()]
        return [top[k - 1] for k in ks]
    med = hub_in.approxQuantile("deg", [0.5], 1e-4) if n_h else []
    deg_q = (med + cuts("deg")) if n_h else []
    str_q = ([None] + cuts("str")) if n_h else []
    hub_in.unpersist()
    def tier(col, cuts):
        t = F.lit(0)
        for i, c in enumerate(cuts):
            t = F.when(F.col(col) >= F.lit(c), F.lit(i + 1)).otherwise(t)
        return t
    fi_ent = fistate.groupBy("ent").agg(
        F.sum(F.col("amt_now").isNotNull().cast("long")).alias("n_fi"),
        F.expr("max_by(fi_c, amt_now)").alias("fi_dom"),
        (F.max(F.col("amt_now").cast("double")) / F.sum(F.col("amt_now").cast("double"))).alias("top_fi_share"),
        F.sum(F.col("fi_new").cast("long")).alias("fi_new"), F.sum(F.col("fi_lost").cast("long")).alias("fi_lost"))
    uni = (slim.select("ent")
               .unionByName(churn.where((F.col("prev_amt") > 0) | (F.col("lost_cust") > 0)).select("ent"))
               .distinct())
    o = (uni.join(slim, "ent", "left").join(elig, "ent", "left").join(full, "ent", "left")
            .join(fi_ent, "ent", "left").join(LABELS, "ent", "left")
            .withColumn("name_class", F.coalesce("name_class", F.lit("not_in_map"))))
    for d in DIRS:
        o = (o.withColumn(f"amt_{d}", F.coalesce(F.col(f"amt_{d}"), D0))
              .withColumn(f"n_txn_{d}", F.coalesce(F.col(f"n_txn_{d}"), F.lit(0).cast("long")))
              .withColumn(f"n_cust_{d}", F.coalesce(F.col(f"n_cust_{d}"), F.lit(0).cast("long"))))
    elig_c = F.coalesce(F.col("eligible"), F.lit(False))
    o = (o.withColumn("amt_all", (F.col("amt_in") + F.col("amt_out")).cast(DEC))
          .withColumn("n_cust_all", F.coalesce(F.col("n_cust_all"), F.lit(0).cast("long")))
          .withColumn("n_ids_active", F.coalesce(F.col("n_ids_active"), F.lit(0).cast("long")))
          .withColumn("eligible", elig_c)
          .withColumn("net_in", F.when(elig_c, F.col("amt_in") - F.col("amt_out")))
          .withColumn("two_way_ratio", F.when(elig_c, F.least("amt_in", "amt_out").cast("double")
                                              / F.greatest("amt_in", "amt_out").cast("double")))
          .withColumn("ent_id", F.col("ent")))
    for c in ["n_fi", "top_fi_share", "fi_new", "fi_lost"]:
        o = o.withColumn(c, F.when(elig_c, F.coalesce(F.col(c), F.lit(0)) if c != "top_fi_share" else F.col(c)))
    if deg_q and len(deg_q) == len(HUB_Q) + 1 and deg_q[0]:
        o = (o.withColumn("hub_tier", F.when(elig_c, F.greatest(tier("n_cust_all", deg_q[1:]), tier("amt_all", str_q[1:]))))
              .withColumn("deg_x_median", F.when(elig_c, F.col("n_cust_all") / F.lit(deg_q[0])))
              .withColumn("hub_med15", F.when(elig_c, F.col("n_cust_all") / F.lit(deg_q[0]) >= HUB_MED_MULT)))
    else:
        o = (o.withColumn("hub_tier", F.when(elig_c, F.lit(0))).withColumn("deg_x_median", F.lit(None).cast("double"))
              .withColumn("hub_med15", F.when(elig_c, F.lit(False))))
    for c in ZERO_COLS:
        o = o.withColumn(c, F.when(elig_c, F.coalesce(F.col(c), F.lit(0))))
    typ = dict(zip(DICT_DF.column, DICT_DF.type))
    o = o.select(*[F.col(c).cast(typ[c]).alias(c) for c in OUT_COLS])

    stage = work_path(m, "_stage_out")
    o.repartition(OUT_FILES).write.mode("overwrite").parquet(stage)
    got = reconcile(m, spark.read.parquet(stage))
    touch(f"{stage}/{SIG_MARK}")
    delete(out_path(m))
    mkdirs(P_OUT)
    rename(stage, out_path(m))
    ensure_table(m)
    not_in_map = idrel.where(~F.col("in_map")).select("cpty_id").distinct().count()
    return {"month": m, "rows": got["rows"], "eligible": got["elig"], "$B": float(got["amt"] or 0) / 1e9,
            "% $ eligible": 100 * float(got["amt_elig"] or 0) / max(float(got["amt"] or 0), 1e-9),
            "ids not in map": not_in_map,
            "hub deg P99/P99.9/P99.99": "/".join(f"{v:,.0f}" for v in deg_q[1:]) if deg_q else "",
            "median deg": deg_q[0] if deg_q else None}
print("step functions ready")

In [ ]:
# §3b  Build loop (rerun this cell to resume)
if REBUILD_FROM and PLAN:
    for m in [x for x in ALL_MONTHS if x >= REBUILD_FROM]:
        delete(out_path(m))
        delete(f"{P_WORK}/{m}")
        for kind in ["rel", "fi"]:
            delete(state_path(kind, m))
    if table_exists(T_OUT):
        for m in [x for x in ALL_MONTHS if x >= REBUILD_FROM]:
            spark.sql(f"ALTER TABLE {T_OUT} DROP IF EXISTS PARTITION (month='{m}')")
    REBUILD_FROM = None          # deleted once; a rerun of this cell resumes normally

PLAN = [m for m in PLAN if month_status(m) != "done"]       # months finished by an earlier attempt are skipped
for m in ALL_MONTHS:                                         # scratch left by a stop after publishing
    if month_status(m) == "done" and m not in PLAN:
        delete(f"{P_WORK}/{m}")
sync_partitions()

RUN = []
handle = display(render(note("starting…")), display_id=True)
def on_step(m, step):
    done = pd.DataFrame(RUN) if RUN else None
    handle.update(render(f"§3 Building {m}: step {step}  ({len(RUN)} of {len(PLAN)} months done this run)",
                         ("Months built this run", done) if done is not None else None))

t_run = time.time()
for m in PLAN:
    t0 = time.time()
    rec = build_month(m, on_step)
    rec["min"] = (time.time() - t0) / 60
    rec["sig"] = SIG
    rec["built_at"] = dt.datetime.now().strftime("%Y-%m-%d %H:%M")
    rec["steps reused"] = sum(1 for s in STEP_LOG if s["month"] == m and s["action"] == "reused")
    RUN.append(rec)
    pd.DataFrame([rec]).to_csv(RUNLOG, mode="a", header=not os.path.exists(RUNLOG), index=False)
    prune_state(m)
    delete(f"{P_WORK}/{m}")

STATUS = {m: month_status(m) for m in ALL_MONTHS}
left = [m for m in ALL_MONTHS if STATUS[m] != "done"]
handle.update(render(
    f"§3 Done: {len(RUN)} month(s) built in {(time.time() - t_run) / 60:.1f} min",
    ("Months built this run", pd.DataFrame(RUN)) if RUN else None,
    note(f"Months still to build: {len(left)}" + (f" ({left[0]} → {left[-1]}); run §1 and §3 again." if left else ".")),
    note(f"Impala: INVALIDATE METADATA {T_OUT};  (afterwards REFRESH {T_OUT}; after each run)")))

## §4 Dictionary and run log

In [ ]:
# §4  Write the dictionary table and show the run log
stage = f"{P_WORK}/_stage_dict"
spark.createDataFrame(DICT_DF.assign(version=METRICS_VERSION)).coalesce(1).write.mode("overwrite").parquet(stage)
delete(P_DICT)
rename(stage, P_DICT)
spark.sql(f"DROP TABLE IF EXISTS {T_DICT}")
cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in spark.read.parquet(P_DICT).schema.fields)
spark.sql(f"CREATE EXTERNAL TABLE {T_DICT} ({cols}) STORED AS PARQUET LOCATION '{P_DICT}'")
log = pd.read_csv(RUNLOG) if os.path.exists(RUNLOG) else pd.DataFrame()
fam = DICT_DF.groupby(["family", "set"]).size().reset_index(name="columns")
show(f"§4 {T_DICT}: {len(DICT_DF)} columns",
     ("Columns by family", fam),
     ("Run log (latest build per month)", log.drop_duplicates("month", keep="last").tail(40) if len(log) else None))

## §5 Sanity views of the latest built month

In [ ]:
# §5  Latest month: population, top entities, hubs, NULL rates of the full set
STATUS = {m: month_status(m) for m in ALL_MONTHS}
built = [m for m in ALL_MONTHS if STATUS[m] == "done"]
assert built, "no month built yet"
LM = built[-1]
o = spark.read.parquet(out_path(LM))
pop = (o.groupBy("eligible", "name_class")
        .agg(F.count("*").alias("entities"), F.sum(F.col("amt_all").cast("double")).alias("amt"))
        .toPandas().sort_values(["eligible", "amt"], ascending=[False, False]))
pop["$B"] = pop.amt / 1e9
cols_top = ["ent_name", "name_class", "cpty_type", "fi_dom", "amt_in", "amt_out", "n_cust_in", "n_cust_out",
            "n_cust_tm_in", "top1_share_in", "retention_in", "new_cust_in", "lost_cust_in", "hub_tier"]
top = o.where("eligible").orderBy(F.desc("amt_all")).select(*cols_top).limit(20).toPandas()
hubs = (o.where("hub_tier >= 2").orderBy(F.desc("n_cust_all"))
         .select("ent_name", "name_class", "n_cust_all", "amt_all", "hub_tier", "deg_x_median", "dep_wavg_in")
         .limit(20).toPandas())
for d_ in (top, hubs):
    for c in ["amt_in", "amt_out", "amt_all"]:
        if c in d_:
            d_[c] = d_[c].astype(float)
el = o.where("eligible")
n_el = el.count()
nulls = el.select(*[F.sum(F.col(c).isNull().cast("long")).alias(c) for c in FULL_COLS]).toPandas().T.reset_index()
nulls.columns = ["column", "null rows"]
nulls["% null (eligible)"] = 100 * nulls["null rows"] / max(n_el, 1)
show(f"§5 {LM}: {o.count():,} entity rows, {n_el:,} eligible",
     ("Population", pop[["eligible", "name_class", "entities", "$B"]]),
     ("Top 20 eligible entities by $", top),
     ("Hubs (tier ≥ 2) by degree", hubs),
     ("Full-set columns with the most NULLs among eligible rows (NULL = no activity in that direction or below a minimum)",
      nulls.sort_values("% null (eligible)", ascending=False).head(15)))

## §6 Housekeeping

In [ ]:
# §6  Folders of other configurations, and the Impala statements
others = sorted(set(ls_dirs(P_STATE_ROOT) + ls_dirs(P_WORK_ROOT)) - {SIG})
sizes = [{"signature": s, "state GB": path_gb(f"{P_STATE_ROOT}/{s}"), "scratch GB": path_gb(f"{P_WORK_ROOT}/{s}")}
         for s in others]
if CLEAN_OTHER_SIGS:
    for s in others:
        delete(f"{P_STATE_ROOT}/{s}")
        delete(f"{P_WORK_ROOT}/{s}")
show("§6 Housekeeping",
     ("Other configuration folders" + (" (deleted)" if CLEAN_OTHER_SIGS else ""), pd.DataFrame(sizes)) if sizes else None,
     note(f"Current: state {path_gb(P_STATE):.1f} GB, scratch {path_gb(P_WORK):.1f} GB, output {path_gb(P_OUT):.1f} GB."),
     note(f"Impala: INVALIDATE METADATA {T_ENT_DIM}; INVALIDATE METADATA {T_OUT}; INVALIDATE METADATA {T_DICT};"))